# Notebook 07 — Fraud Detection Platform: BCBS 239 Data Governance
**Master Playbook / gap-analysis priority 8 — a real, rerunnable data-quality gate (nulls, duplicates, dtype/range conformance, row count, append-only history mirroring NB3's pattern) plus an honest mapping of real project evidence against the 6 BCBS 239 principle-groups this project's own gap-analysis doc already sourced. Reuses, does not recompute, NB2's real concentration report — and discloses plainly, rather than papering over, the two principle-groups this anonymized single-table dataset genuinely cannot satisfy.**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment (thread ceiling set BEFORE any import)
# CPU/RAM thresholds: CPU 93% (90-95% band), RAM 90%.
# ============================================================
import os, time, json, warnings, subprocess, sys
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("POLARS_MAX_THREADS", str(_N_THREADS))

# Lean auto-install guard -- only what this notebook needs: Polars for the
# fast real CSV load, psutil for RAM reporting, pyarrow as Polars' backend.
# No ML imports at all (no model load, no scoring) -- this is a data-
# governance notebook, not a modeling one.
for _pkg in ("polars", "psutil", "pyarrow"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import polars as pl
import psutil
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
print("Setup complete.")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1-06.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

REPORTS_DIR = os.path.join(REPO_ROOT, "reports")
RESULTS_DIR = os.path.join(REPORTS_DIR, "nb7_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb7_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:      {REPO_ROOT}")
print(f"NB7 results in: {RESULTS_DIR}")

##############################################################################
# LOAD NB1's REAL EXPECTED ROW COUNT + NB6's REAL TIER -- no recomputation.
##############################################################################
with open(os.path.join(REPORTS_DIR, "nb1_results", "nb1_final_results.json"), encoding="utf-8") as f:
    nb1_results = json.load(f)
EXPECTED_ROW_COUNT = nb1_results["dataset"]["rows"]
NB1_DUPLICATE_COUNT_REPORTED = nb1_results["dataset"]["duplicates"]["n_duplicate_rows"]

_nb6_tier_path = os.path.join(REPORTS_DIR, "nb6_results", "nb6_model_tiering_matrix.json")
nb6_tier = None
if os.path.exists(_nb6_tier_path):
    with open(_nb6_tier_path, encoding="utf-8") as f:
        nb6_tier = json.load(f)

_nb2_report_path = os.path.join(REPORTS_DIR, "nb2_results", "nb2_validation_report.json")
nb2_gate1_status = None
if os.path.exists(_nb2_report_path):
    with open(_nb2_report_path, encoding="utf-8") as f:
        nb2_gate1_status = json.load(f).get("gate1_all_passed")

print(f"Expected row count (real, from NB1): {EXPECTED_ROW_COUNT:,}")
print(f"NB1's originally reported duplicate-row count: {NB1_DUPLICATE_COUNT_REPORTED}")
print(f"NB6 real tier on file: {nb6_tier['model_tier'] if nb6_tier else 'not found -- run NB6 first for the Governance mapping to cite a real tier'}")

##############################################################################
# DATA_PATH resolution + Polars-accelerated load -- NB1-06's fixes reused
# unchanged.
##############################################################################
DATA_PATH = None
for _cand in [
    r"C:\Users\rnand\Downloads\creditcard.csv\creditcard.csv",  # your real dataset location -- checked first
    "creditcard.csv",
    os.path.join("data", "raw", "creditcard.csv"),
    os.path.join("..", "data", "raw", "creditcard.csv"),
    os.path.join("..", "creditcard.csv"),
]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv not found. Place it next to this notebook, or at "
        "data/raw/creditcard.csv relative to the repo root."
    )
print("Using DATA_PATH:", DATA_PATH)

_SCHEMA_OVERRIDES = {"Time": pl.Float64, "Amount": pl.Float64, "Class": pl.Int64}
for _i in range(1, 29):
    _SCHEMA_OVERRIDES[f"V{_i}"] = pl.Float64

_t_load0 = time.time()
df = pl.read_csv(DATA_PATH, schema_overrides=_SCHEMA_OVERRIDES).to_pandas()
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols via Polars in {time.time()-_t_load0:.3f}s")
_ram_after_load = psutil.virtual_memory()
print(f"RAM after load: {_ram_after_load.percent:.1f}% used ({_ram_after_load.used/1e9:.2f} GB / {_ram_after_load.total/1e9:.2f} GB)")

V_COLS = [f"V{i}" for i in range(1, 29)]

print("=" * 70)
print("SECTION A -- REAL, RERUNNABLE DATA-QUALITY GATE (append-only, mirrors NB3's monitoring pattern)")
print("=" * 70)
print("  Note: rerunning this against the SAME static historical file will correctly report the same real "
      "result every time -- that consistency is itself a legitimate signal (confirms nothing silently changed "
      "in the source file). This gate is meant to be rerun against each NEW incoming real data batch in "
      "production; today's run establishes the real baseline entry.")

def run_data_quality_check(_df, _expected_rows):
    """Real, vectorized data-quality checks -- no ML, no scoring, pure pandas/numpy on the real data."""
    _checks = {}
    _null_counts = _df.isna().sum()
    _checks["no_nulls_anywhere"] = bool(int(_null_counts.sum()) == 0)
    _checks["amount_non_negative"] = bool((_df["Amount"] >= 0).all())
    _checks["time_non_negative"] = bool((_df["Time"] >= 0).all())
    _checks["class_is_binary"] = bool(set(_df["Class"].unique()).issubset({0, 1}))
    _numeric_block = _df[V_COLS + ["Amount", "Time"]].to_numpy()
    _checks["all_numeric_features_finite"] = bool(np.isfinite(_numeric_block).all())
    _checks["row_count_matches_expected"] = bool(len(_df) == _expected_rows)

    _dup_count = int(_df.duplicated().sum())
    _dup_rate = _dup_count / len(_df)

    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "n_rows": int(len(_df)),
        "expected_rows": int(_expected_rows),
        "checks": _checks,
        "all_checks_passed": bool(all(_checks.values())),
        "null_counts_per_column": {k: int(v) for k, v in _null_counts.items() if v > 0},
        "duplicate_row_count_fresh": _dup_count,
        "duplicate_row_rate_fresh": _dup_rate,
        "duplicate_row_count_nb1_originally_reported": int(NB1_DUPLICATE_COUNT_REPORTED),
        "duplicate_counts_match_nb1": bool(_dup_count == NB1_DUPLICATE_COUNT_REPORTED),
        "column_stats": {
            "Time": {"min": float(_df["Time"].min()), "max": float(_df["Time"].max()),
                     "mean": float(_df["Time"].mean()), "std": float(_df["Time"].std())},
            "Amount": {"min": float(_df["Amount"].min()), "max": float(_df["Amount"].max()),
                       "mean": float(_df["Amount"].mean()), "std": float(_df["Amount"].std())},
        },
    }

_t_dq0 = time.time()
dq_entry = run_data_quality_check(df, EXPECTED_ROW_COUNT)
_dq_seconds = time.time() - _t_dq0

_history_path = os.path.join(RESULTS_DIR, "data_quality_history.jsonl")
with open(_history_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(dq_entry, default=str) + "\n")

for _k, _v in dq_entry["checks"].items():
    print(f"  [{'PASS' if _v else 'FAIL'}] {_k}")
print(f"  Fresh duplicate-row count: {dq_entry['duplicate_row_count_fresh']} ({dq_entry['duplicate_row_rate_fresh']:.4%} of rows) "
      f"-- matches NB1's originally reported {NB1_DUPLICATE_COUNT_REPORTED}: {dq_entry['duplicate_counts_match_nb1']}")
print(f"  ALL CHECKS PASSED: {dq_entry['all_checks_passed']}")
print(f"  Data-quality gate computed in {_dq_seconds:.3f}s. Appended to: {_history_path}")

print("=" * 70)
print("SECTION B -- CONCENTRATION / COMPLETENESS (reusing NB2's real report, not recomputed)")
print("=" * 70)
_concentration_path = os.path.join(REPORTS_DIR, "nb2_results", "concentration_report.csv")
concentration_summary = None
if os.path.exists(_concentration_path):
    _conc = pd.read_csv(_concentration_path)
    concentration_summary = {
        "source_file": _concentration_path,
        "n_segments": int(len(_conc)),
        "max_false_positive_rate": float(_conc["false_positive_rate"].max()),
        "max_false_negative_rate": float(_conc["false_negative_rate"].max()),
        "segment_with_max_fp_rate": _conc.loc[_conc["false_positive_rate"].idxmax(), ["amount_band", "hour_of_day"]].to_dict(),
    }
    print(f"  Loaded NB2's real concentration report: {concentration_summary['n_segments']} real Amount-band x hour-of-day segments")
    print(f"  Worst real segment FP rate: {concentration_summary['max_false_positive_rate']:.4%} "
          f"at {concentration_summary['segment_with_max_fp_rate']}")
else:
    print(f"  NB2's concentration_report.csv not found at {_concentration_path} -- run NB2 first.")
print("  Disclosed limitation: BCBS 239's Completeness principle calls for risk data viewable by business line, "
      "legal entity, asset type, industry, and region. This anonymized, single-source, single-table consumer-"
      "transaction dataset (V1-V28 are PCA components) has NO such dimensions -- there is no fabricated substitute "
      "here. Amount-band x hour-of-day (NB2's real segment cuts, reused above) is the closest real, computable "
      "analogue available at this project's scope, and is disclosed as a substitute, not a full satisfaction, "
      "of the Completeness principle.")

print("=" * 70)
print("SECTION C -- BCBS 239 PRINCIPLE-GROUP MAPPING (real project evidence only)")
print("=" * 70)

_tier_note = (
    f"NB6 real Tier {nb6_tier['model_tier']} ({nb6_tier['tier_description']})" if nb6_tier
    else "NB6 not yet run -- no real tier on file to cite"
)

bcbs239_mapping = [
    {"principle_group": "Governance", "status": "evidenced",
     "evidence": f"{_tier_note}; oversight cadence follows NB6's real recommended-next-steps for that tier."},
    {"principle_group": "Data architecture", "status": "limitation_disclosed",
     "evidence": "Single anonymized source table (V1-V28 PCA components, Amount, Time, Class) -- no cross-source "
                 "identifiers or taxonomy exist to integrate at this project's scope. Not fabricated."},
    {"principle_group": "Accuracy & integrity", "status": "evidenced",
     "evidence": f"NB2 Gate 1 structural checks (real, all_passed={nb2_gate1_status}); this notebook's own real, "
                 f"rerunnable data-quality gate (Section A, all_checks_passed={dq_entry['all_checks_passed']})."},
    {"principle_group": "Completeness", "status": "limitation_disclosed",
     "evidence": "No business-line/legal-entity/region dimensions exist in this dataset (see Section B). "
                 "Amount-band x hour-of-day (NB2's real concentration report) is the disclosed substitute."},
    {"principle_group": "Timeliness", "status": "evidenced",
     "evidence": "NB3's real append-only monitoring cadence + this notebook's own real append-only data-quality "
                 "gate -- both designed to be rerun against each new real data batch, aggregation frequency "
                 "scaling with how often new data arrives."},
    {"principle_group": "Adaptability", "status": "evidenced",
     "evidence": "NB6's real retroactive finding (recommending NB3's monitoring tier parameter be updated from "
                 "its default) is a live, real example of the system adapting to a new governance decision."},
]
for _row in bcbs239_mapping:
    print(f"  [{_row['status']:>20s}] {_row['principle_group']}: {_row['evidence'][:100]}...")

##############################################################################
# RENDER THE WRITTEN BCBS 239 GOVERNANCE MAPPING (real numbers only)
##############################################################################
_generated_at = datetime.now(timezone.utc).isoformat()
_mapping_rows_md = "\n".join(
    f"| {_r['principle_group']} | {_r['status'].replace('_', ' ')} | {_r['evidence']} |" for _row_i, _r in enumerate(bcbs239_mapping)
)
_checks_md = "\n".join(f"- [{'x' if _v else ' '}] {_k}" for _k, _v in dq_entry["checks"].items())

_md = f"""# BCBS 239 Data Governance — Fraud Detection Platform

**Generated:** {_generated_at}
**Real data-quality gate: ALL CHECKS PASSED = {dq_entry['all_checks_passed']}**

## Section A — Real, rerunnable data-quality gate
{_checks_md}

Fresh duplicate-row count: {dq_entry['duplicate_row_count_fresh']} ({dq_entry['duplicate_row_rate_fresh']:.4%}) — matches NB1's originally reported {NB1_DUPLICATE_COUNT_REPORTED}: {dq_entry['duplicate_counts_match_nb1']}

## Section B — Concentration / Completeness
{"Loaded NB2's real concentration report: " + str(concentration_summary['n_segments']) + " real segments, worst FP rate " + format(concentration_summary['max_false_positive_rate'], '.4%') if concentration_summary else "NB2's concentration report not found."}

**Disclosed limitation:** this anonymized, single-source dataset has no business-line/legal-entity/region dimensions — Amount-band x hour-of-day is the disclosed real substitute, not a full satisfaction of BCBS 239's Completeness principle.

## Section C — BCBS 239 principle-group mapping (project's own sourced summary; real evidence only)

| Principle group | Status | Evidence |
|---|---|---|
{_mapping_rows_md}

## Scope note
This maps against the 6 principle-groups this project's own gap-analysis doc already sourced from BCBS 239 (Governance, Data architecture, Accuracy & integrity, Completeness, Timeliness, Adaptability) — not a hand-typed reproduction of the full 14-principle regulatory text, to avoid misstating exact regulatory wording this portfolio project has not independently sourced number-by-number.
"""

_html = "<div class='bcbs239-mapping'>" + "".join(
    f"<h1>{_l[2:]}</h1>" if _l.startswith("# ") else
    f"<h2>{_l[3:]}</h2>" if _l.startswith("## ") else
    f"<p>{_l}</p>" if _l.strip() and not _l.startswith("|") and not _l.startswith("-") else
    (f"<li>{_l[2:]}</li>" if _l.startswith("- ") else "")
    for _l in _md.splitlines()
) + "</div>"

with open(os.path.join(RESULTS_DIR, "bcbs239_governance_mapping.md"), "w", encoding="utf-8") as f:
    f.write(_md)
with open(os.path.join(RESULTS_DIR, "bcbs239_governance_mapping.html"), "w", encoding="utf-8") as f:
    f.write(_html)

##############################################################################
# SAVE NOTEBOOK 07 RESULTS
##############################################################################
nb7_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS, "generated_at_utc": _generated_at},
    "data_quality_gate": dq_entry,
    "concentration_summary": concentration_summary,
    "bcbs239_mapping": bcbs239_mapping,
}
with open(os.path.join(RESULTS_DIR, "nb7_bcbs239_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb7_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB) -- "
      f"stayed under the {RAM_THRESHOLD_PCT}% ceiling: {_ram_end.percent < RAM_THRESHOLD_PCT}")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print(f"Notebook 07 complete. Results written to: {os.path.join(RESULTS_DIR, 'nb7_bcbs239_report.json')}")
print("Written: bcbs239_governance_mapping.md, bcbs239_governance_mapping.html, data_quality_history.jsonl (appended)")
